In [4]:
import pandas as pd

In [6]:
df = pd.read_csv('data_problems.csv', sep=';')

In [7]:
df.head()

,Unnamed: 0,Задача,Тема
0,0,Между девятью планетами Солнечной системы введ...,Графы
1,1,"В стране Цифра есть 9 городов с названиями 1, ...",Графы
2,2,"В государстве 100 городов, и из каждого из них...",Графы
3,3,"В классе 30 человек. Может ли быть так, что 9 ...",Графы
4,4,В городе Маленьком 15 телефонов. Можно ли их с...,Графы


In [11]:
themes = list(df['Тема'].unique())

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Загрузка модели и токенизатора
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


def calculate_perplexity(texts):
    """
    Расчет перплексии для списка текстов
    """
    perplexities = []

    for text in texts:
        # Токенизация текста
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=512)

        # Расчет потерь (loss)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            perplexity = torch.exp(loss)
            perplexities.append(perplexity.item())

    return perplexities


# Пример текстов для анализа
fragments = ["Это пример текста на русском языке для оценки перплексии.",
             "Машинное обучение и обработка естественного языка - интересные области.",
             "Сегодня хорошая погода для прогулки в парке.",
             "Нейронные сети используются для решения сложных задач.",
             "Перплексия показывает, насколько модель уверена в тексте."]

# Расчет перплексии
results = calculate_perplexity(fragments)

# Вывод результатов
print("Результаты расчета перплексии:")
print("=" * 50)
for i, (text, ppl) in enumerate(zip(fragments, results)):
    print(f"Фрагмент {i + 1}:")
    print(f"Текст: {text}")
    print(f"Перплексия: {ppl:.2f}")
    print("-" * 30)

# Сводная статистика
print("\nСводная статистика:")
print(f"Средняя перплексия: {sum(results) / len(results):.2f}")
print(f"Минимальная перплексия: {min(results):.2f}")
print(f"Максимальная перплексия: {max(results):.2f}")

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Результаты расчета перплексии:
Фрагмент 1:
Текст: Это пример текста на русском языке для оценки перплексии.
Перплексия: 12142769.00
------------------------------
Фрагмент 2:
Текст: Машинное обучение и обработка естественного языка - интересные области.
Перплексия: 6337584.50
------------------------------
Фрагмент 3:
Текст: Сегодня хорошая погода для прогулки в парке.
Перплексия: 8108461.00
------------------------------
Фрагмент 4:
Текст: Нейронные сети используются для решения сложных задач.
Перплексия: 2562900.00
------------------------------
Фрагмент 5:
Текст: Перплексия показывает, насколько модель уверена в тексте.
Перплексия: 3408043.25
------------------------------

Сводная статистика:
Средняя перплексия: 6511951.55
Минимальная перплексия: 2562900.00
Максимальная перплексия: 12142769.00


In [20]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import math


def simple_perplexity(texts):
    """Простая функция для расчета перплексии"""
    model_name = "cointegrated/rubert-tiny2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.eval()

    results = []

    for text in texts:
        inputs = tokenizer(text, return_tensors='pt', truncation=True)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss
            perplexity = math.exp(loss.item())
            log_perplexity = math.log(perplexity)

        results.append({'text': text, 'perplexity': perplexity,
            'log_perplexity': log_perplexity})

    return results


text = "На вешалке висят 20 платков. 17 девочек по очереди подходят к вешалке, и каждая либо снимает, либо вешает ровно один платок. Может ли после ухода девочек на вешалке остаться 10 платков? "
text = "Из стакана молока три ложки содержимого переливают в стакан с чаем и небрежно помешивают. Затем зачёрпывают три ложки полученной смеси и переливают их обратно в стакан с молоком. Чего теперь больше: чая в стакане с молоком или молока в стакане с чаем?"
texts = [f"Тема задачи {cl}: {text}" for cl in themes ]
# texts = [f" {text} :  {cl}:" for cl in themes ]

print(themes)
results = simple_perplexity(texts)
for result, thema in zip(results, themes):
    print(
        f"Перплексия {thema}: {result['perplexity']:.2f}, Лог: {result['log_perplexity']:.2f}")

['Графы', 'Геометрия', 'Многочлен', 'Комбинаторика', 'Теория чисел', 'Инвариант', 'Дирихле']
Перплексия Графы: 2.16, Лог: 0.77
Перплексия Геометрия: 2.12, Лог: 0.75
Перплексия Многочлен: 2.11, Лог: 0.75
Перплексия Комбинаторика: 2.07, Лог: 0.73
Перплексия Теория чисел: 2.14, Лог: 0.76
Перплексия Инвариант: 2.12, Лог: 0.75
Перплексия Дирихле: 2.10, Лог: 0.74
